In [4]:
# Student Feedback Classification

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Load dataset

file_name = "student_feedback_experiments_1_to_5 (1) (1).csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())


# Clean column names

df.columns = df.columns.str.strip()


# Remove missing values

df = df.dropna(subset=["Feedback", "Category"])

print("\nDataset after removing missing values:")
print(df.shape)


# Input and target

X = df["Feedback"]
y = df["Category"]


# Check category counts

print("\nCategory distribution:")
print(y.value_counts())


# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# TF-IDF vectorization

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF conversion completed!")
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)


# Create models

models = {
    "Naive Bayes": MultinomialNB(),
    "SVM": LinearSVC()
}


# Train and evaluate models

results = []

for name, model in models.items():

    model.fit(X_train_tfidf, y_train)

    predictions = model.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    print("\n", name)
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))


# Comparison table

results_df = pd.DataFrame(results)

print("\nFinal Model Comparison")
print(results_df.to_string(index=False))


# Find best model

best_model = results_df.loc[
    results_df["Accuracy"].idxmax()
]

print("\nBest Model:", best_model["Model"])
print("Best Accuracy:", round(best_model["Accuracy"], 4))


Dataset loaded successfully!
Shape: (30, 4)

Columns:
['Feedback_ID', 'Feedback', 'Category', 'Sentiment']

First 5 rows:
   Feedback_ID                                           Feedback  \
0            1  Faculty explains concepts clearly and effectiv...   
1            2             Hostel food quality needs improvement.   
2            3       Placement training sessions are very useful.   
3            4                 Laboratory computers are outdated.   
4            5                   Library resources are excellent.   

         Category Sentiment  
0        Academic  Positive  
1          Hostel  Negative  
2       Placement  Positive  
3  Infrastructure  Negative  
4        Academic  Positive  

Dataset after removing missing values:
(30, 4)

Category distribution:
Category
Academic          8
Placement         8
Hostel            7
Infrastructure    7
Name: count, dtype: int64

Training samples: 24
Testing samples: 6

TF-IDF conversion completed!
Training TF-IDF shape: (2